## 🔧 Environment Setup & Checkpoint Loading

_Added to fix the cross-notebook data-flow issue: this cell mounts Drive, imports the common libraries this notebook needs, and defines helper functions to save/load intermediate results so this notebook works correctly whether it's run right after the previous one or on its own, days later, after the raw data has changed._

In [ ]:
# 🔧 SETUP: local paths, common imports & checkpoint utilities
# (Local/VS Code version — no Google Drive here. If you actually run these
# notebooks in Google Colab instead, use the "_FIXED" versions, not "_LOCAL".)
#
# This cell fixes the "notebook 2 can't see notebook 1's data" problem: instead
# of relying on variables still sitting in memory from another notebook (which
# never works once notebooks are split into separate .ipynb files, or run in
# separate VS Code kernels), every notebook loads what it needs from a shared
# "checkpoints" folder on disk, and saves what later notebooks need back to
# that same folder.

import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# 📁 Data lives in a `data` folder that's a *sibling* of the folder these
# notebooks are run from (e.g. `project/notebooks/*.ipynb` + `project/data/`).
# This resolves the same way no matter whose machine it runs on - no
# usernames or absolute paths baked in. Adjust the '..' below if your data
# folder sits somewhere else relative to your notebooks.
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
DATA_DIR = BASE_DIR
CKPT_DIR = os.path.join(BASE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)


def save_ckpt(obj, name):
    """Save a variable to the shared checkpoint folder so later notebooks can load it."""
    path = os.path.join(CKPT_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"✅ Saved checkpoint '{name}' -> {path}")


def load_ckpt(name):
    """Load a variable that was saved by an earlier notebook in this pipeline."""
    path = os.path.join(CKPT_DIR, f'{name}.pkl')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ Checkpoint '{name}' not found at {path}.\n"
            f"   Make sure you have run the earlier notebook(s) in the pipeline first "
            f"(they save this checkpoint at their final cell), and that BASE_DIR "
            f"above points at the right folder."
        )
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    print(f"✅ Loaded checkpoint '{name}' <- {path}")
    return obj


## 📥 Load Checkpoints From Earlier Notebooks

In [ ]:
# 📥 Load the variables this notebook needs from earlier notebooks in the pipeline
df = load_ckpt('part1_df')
sales = load_ckpt('sales')
stores = load_ckpt('stores')
features = load_ckpt('features_raw')


## 📈 Store Type Sales Comparison

- **Store Type A** leads in total weekly sales, followed by Type B, with Type C trailing.
- This suggests Type A stores drive the majority of revenue; investigate what makes Type A successful (e.g., location, size, product mix).
- For Type B and C, consider tailored strategies (promotions, layout changes) to boost performance or optimize operations.


## 🔍 Findings: Store Performance Analysis

1. **🏪 Store Type Performance**
   - **Store Type A** recorded the **highest earnings**, followed closely by **Store Type B**.
   - **Store Type C** consistently showed the **lowest weekly sales**, indicating lower revenue potential or smaller market coverage.

2. **📉 Store Characteristics and Sales Trends**
   - **Store Types A and B** experienced a **slight year-over-year decline in sales (~5%)**, despite having a higher number of stores.
   - This may suggest **market saturation** or the need for **strategy optimization** in these store types.

3. **📈 Store Type C Stability**
   - **Store Type C**, while generating lower sales overall, displayed **stable performance** and a **modest annual growth of approximately 3%**.
   - This suggests that **Store Type C may serve a niche market** or benefit from


## Departments with highest sales

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Aggregate total sales per department and convert to millions
dept_sales = df.groupby('Dept', as_index=False)['Weekly_Sales'].sum()
dept_sales['Weekly_Sales_Million'] = dept_sales['Weekly_Sales'] / 1_000_000

# 2. Sort departments by sales descending
dept_sales_sorted = dept_sales.sort_values('Weekly_Sales_Million', ascending=False)

# 3. Choose how many top departments to display (e.g., top 20) for clarity
top_n = 20
top_depts = dept_sales_sorted.head(top_n).copy()

# 4. Set style
sns.set_style('whitegrid')
plt.figure(figsize=(12,8))

# 5. Plot horizontal bar chart for better label readability
ax = sns.barplot(
    y=top_depts['Dept'].astype(str),
    x=top_depts['Weekly_Sales_Million'],
    palette='rocket'
)

# 6. Labels and title
ax.set_xlabel('Total Weekly Sales (Millions)', fontsize=12)
ax.set_ylabel('Department', fontsize=12)
ax.set_title(f'📊 Top {top_n} Departments by Sum of Weekly Sales', fontsize=14, fontweight='bold')

# 7. Annotate values on bars
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{width:.1f}",
                xy=(width, p.get_y() + p.get_height() / 2),
                xytext=(5, 0),
                textcoords='offset points',
                ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 📈 Top Departments by Total Weekly Sales

- Displaying the top 20 departments by cumulative weekly sales (in millions).
- Departments 38, 92, and 95 (among others in the top list) show the highest sales volumes.
- **Insight:** Focus inventory planning, promotions, and cross-selling strategies on these high-performing departments.  
- For lower-performing departments, analyze whether adjustments in merchandising, pricing, or marketing could boost their performance.

> 💡 **Next Steps:**  
> - Drill down further: analyze monthly or seasonal performance within these top departments.  
> - Investigate if certain store types or regions drive these department sales disproportionately.  
> - Combine with Market Basket Analysis to see what other departments co-occur with these top sellers for cross-sell opportunities.


In [ ]:
# 5️⃣ EDA & VISUALIZATION
sns.heatmap(df[['Weekly_Sales', 'Size','Temperature','Fuel_Price','CPI','Unemployment']].corr(), annot=True)
plt.show()

## 🧾 Summary Insights

1. **📉 Weak Correlation of Weekly Sales with Other Parameters**  
   - Weekly sales do **not exhibit a strong correlation** with other features such as temperature, fuel price, or CPI.  
   - This indicates that sales performance may be driven more by **seasonality, promotions, or localized factors** than by macroeconomic indicators alone.

2. **📉 CPI vs. Unemployment — Negative Correlation**  
   - There is a **negative correlation between the Consumer Price Index (CPI) and Unemployment**, suggesting that as prices rise (inflation), unemployment tends to fall, which may reflect a growing economy.

3. **🛢️ Fuel Price vs. Unemployment — Negative Correlation**  
   - Similarly, **Fuel Price and Unemployment** also show a **negative correlation**.  
   - This relationship may point toward improved economic activity (higher fuel usage, lower unemployment) during certain periods.

---

> 💡 **Implication:**  
While these correlations offer some macro-level economic insights, **direct relationships with sales are weak**, highlighting the importance of exploring other variables like **markdowns, holidays, store size/type**, and **consumer behavior trends** for accurate forecasting and strategic planning.


# **Data Preprocessing and Feature Engineering:**

Handle missing values, especially in the MarkDown data.

Create new features that could influence sales (e.g., store size/type, regional factors).

## Dataset Inspection

In [ ]:
#Merging all the 3 dataset together
sale_store_df = sales.merge(stores, on = 'Store',how='left')
sale_store_df.shape
retail_df = sale_store_df.merge(features, on = ['Store','Date','IsHoliday'],how='left')
retail_df.shape

In [ ]:
final=retail_df.copy()

In [ ]:
#Check for missing values
final.isnull().sum()

In [ ]:
#Check for duplicate rows
final.duplicated().sum()

In [ ]:
final.info()

In [ ]:
# Convert dates to datetime objects
final['Date'] = pd.to_datetime(final['Date'],format='%d/%m/%Y')

In [ ]:
final.info()

In [ ]:
final.describe()

In [ ]:
final.describe(include=['object'])

### 🛠️ Sales Data Preprocessing & Anomaly Detection Pipeline

This section outlines the complete preprocessing steps for handling and preparing sales data, ensuring optimal performance for anomaly detection models.

#### ✅ Step 1: Handle Missing Values
We impute missing values using the **median** strategy to avoid data skew from extreme outliers.

#### ✅ Step 2: Normalize Sales
We use **Min-Max Scaling** to bring all sales values into the same range, enhancing model training stability.

#### ✅ Step 3: Apply Square Root Transformation
To handle positive skewness in the sales data, we apply a **square root transformation**.

#### ✅ Step 4: Detect Anomalies (Z-score Based)
Anomalies are detected using the Z-score method — any point with a Z-score > |3| is flagged as an outlier.

#### ✅ Step 5: KDE (Violin-style) Plots
KDE plots help visualize the distribution of Markdown variables for additional insights into potential outliers or skewness.

### **KDE plot for MarkDown Features**

In [ ]:
# 📊 Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# 🎨 KDE Plotting for MarkDown Features
features = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
num_features = len(features)
num_cols = 2
num_rows = (num_features + 1) // num_cols

fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(14, 8))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.kdeplot(data=df, x=feature, ax=axes[i], fill=True, color='skyblue')
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].set_xlabel(feature)

# Hide unused subplots if any
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

### **Histogram Visualization for MarkDown Features**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the features to visualize
features = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Calculate rows and columns for subplots
num_features = len(features)
num_cols = 2
num_rows = (num_features + 1) // num_cols  # ensure enough rows

# Create subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(15, 10))

# Flatten axes array for easy indexing
axes = axes.flatten()

# Plot each feature
for i, feature in enumerate(features):
    sns.histplot(data=final, x=feature, kde=True, ax=axes[i], color='skyblue', edgecolor='black')
    axes[i].set_title(f'📊 Histogram of {feature}', fontsize=12)
    axes[i].set_xlabel(feature, fontsize=10)
    axes[i].set_ylabel("Frequency", fontsize=10)

# Remove unused subplots if any
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# Final layout adjustment
plt.suptitle("Distribution of MarkDown Features", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])  # leave space for suptitle
plt.show()


### **Boxplot Visualization for MarkDown Features**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define MarkDown features
features = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Calculate number of rows and columns
num_cols = 2
num_rows = (len(features) + 1) // num_cols

# Create subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(15, 10))
axes = axes.flatten()

# Create boxplots for each feature
for i, feature in enumerate(features):
    sns.boxplot(data=final, y=feature, ax=axes[i], palette='pastel')
    axes[i].set_title(f'📦 Boxplot of {feature}', fontsize=12)
    axes[i].set_ylabel(feature, fontsize=10)

# Remove empty subplots if features are odd
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# Super title and layout
plt.suptitle("Boxplots of MarkDown Features", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### 📊 **Findings:**

- The histogram clearly depicts that the distribution of **MarkDown features** is **right-skewed** and **not normally distributed**.
- The **boxplots** reveal the presence of **outliers** in each of the MarkDown features.
- Approximately **5%–8%** of the total records contain **outlier values**, particularly in the MarkDown-related columns.

---

### 🛠️ **Approach:**

- Removing outliers using **IQR-based thresholds** is **not recommended** in this case, as it could result in the removal of **10% or more** of the dataset, leading to **data loss** and reduced availability for model training.
  
- Instead, we will adopt **robust statistical techniques**, such as:
  - **Median imputation** (rather than mean) for missing values
  - **RobustScaler** (or similar methods) that are less sensitive to outliers during feature scaling

---

### ✅ **Why this approach?**

- **Reduces the impact** of outliers while retaining the majority of the data
- Makes the data distribution more **symmetric**
- Ensures **better model performance** by preserving valuable training data and reducing skew


##**Cleaning the data before anomaly detection and handling.**

### **Step 1️⃣ – Handle Missing Values**

🛠 **Use techniques like median imputation to address missing values in your Markdown data.**

- This step ensures that the dataset is complete and ready for analysis.
- Median imputation is preferred here as it is **robust to outliers** compared to the mean.
- Helps maintain the central tendency of skewed distributions (like the Markdown features).

In [ ]:
import pandas as pd

# List of Markdown features
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Fill missing values using median for each column
for col in markdown_cols:
    if col in final.columns:
        final[col] = final[col].fillna(final[col].median())

### Checking again for null values

In [ ]:
final.isnull().sum()

###**Visualization of Markdowns after handling missing values**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the features to plot
features = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Calculate grid size
num_features = len(features)
num_cols = 2
num_rows = (num_features + 1) // num_cols

# Create subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(14, 8))
axes = axes.flatten()

# KDE Plot for each feature
for i, feature in enumerate(features):
    sns.kdeplot(data=final, x=feature, ax=axes[i], fill=True, color='blue', linewidth=2)
    axes[i].set_title(f'KDE Plot of {feature}', fontsize=11)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Density')
    axes[i].grid(True)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


### ✅ Step 2 – Normalize Sales Data

**Objective:**  
To bring all sales data onto a consistent scale and make it suitable for analysis and machine learning models.

---

**Why Normalize?**  
Normalization is essential when features have different units or magnitudes. It helps in:
- Reducing the impact of outliers  
- Improving convergence in machine learning models  
- Ensuring fair weightage for all variables  

---

**📊 Techniques Used:**

**Standard Scaling (Z-score Normalization):**  
This rescaling method transforms the data so it has a **mean of 0** and **standard deviation of 1**.


In [ ]:
# ✅ Step 2: Normalize Sales Data (Min-Max Scaling)

from sklearn.preprocessing import MinMaxScaler

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Reshape the Weekly_Sales column to 2D and apply scaling
final['Weekly_Sales_normalized'] = scaler.fit_transform(final[['Weekly_Sales']])

# Display basic stats before and after normalization
print("Original Weekly Sales (Min, Max):",
      final['Weekly_Sales'].min(), final['Weekly_Sales'].max())
print("Normalized Weekly Sales (Min, Max):",
      final['Weekly_Sales_normalized'].min(), final['Weekly_Sales_normalized'].max())

### ✅ **Step-3: Square Root Transformation to Handle Skewness**

Handling skewed data distributions is critical in many machine learning and statistical applications. Skewed data refers to distributions that are not symmetrical and deviate from a normal (bell-shaped) curve.

When data is right-skewed (i.e., contains a long tail of large values), it can negatively impact model performance and visual analysis. To address this, we apply **data transformation** techniques.

#### 🔹 Why Square Root Transformation?

- Square root transformation is especially useful for **right-skewed** data with **moderate skewness**.
- It **compresses large values** while maintaining the order and structure of data.
- This helps in **reducing skewness** and bringing the distribution closer to normal.

> ⚠️ Note: Unlike log transformation, square root can be applied even when values are zero (but not negative).

#### 💡 Benefits:
- Makes distribution more symmetric.
- Stabilizes variance.
- Improves the performance of models sensitive to data distribution (e.g., linear regression, clustering).

Next, we'll apply a square root transformation to the `Weekly_Sales` column.

In [ ]:
final['Sales_sqrt'] = final['Weekly_Sales_normalized'].apply(lambda x: x ** 0.5)

In [ ]:
final.head(5)

### **Visualizing the Distribution Before and After Transformation**

In [ ]:
# 🧪 Features to visualize
features = ['Weekly_Sales', 'Weekly_Sales_normalized', 'Sales_sqrt']
num_features = len(features)
num_cols = 2
num_rows = (num_features + 1) // num_cols

# 📊 Set up the subplot layout
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(14, 8))
axes = axes.flatten()

# 📌 Plot histogram with KDE for each feature
for i, feature in enumerate(features):
    sns.histplot(data=final, x=feature, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f"Distribution of {feature}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Frequency")

# Turn off any extra axes if they exist
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


### **Comparison of Weekly Sales Distribution Before and After Square Root Transformation**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set style for better visuals
plt.style.use('ggplot')
plt.figure(figsize=(14, 5))

# 📌 Original Data Distribution
plt.subplot(1, 2, 1)
plt.hist(final['Weekly_Sales'], bins=50, color='salmon', edgecolor='black')
plt.xlabel("Weekly Sales")
plt.ylabel("Frequency")
plt.title("Original Distribution of Weekly Sales")

# 📌 Transformed Data Distribution (Square Root)
plt.subplot(1, 2, 2)
plt.hist(final['Sales_sqrt'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Square Root Transformed Sales")
plt.ylabel("Frequency")
plt.title("Distribution After Square Root Transformation")

plt.tight_layout()
plt.show()

## 💾 Save Checkpoints For Next Notebook(s)



In [ ]:
# 💾 Save the variables later notebooks in the pipeline will need
save_ckpt(final, 'part2_final')
